# MotorCam – fáze 3: detektor děr (YOLO)
Nejmenší **YOLO11n** (ultralytics) předtrénovaný na COCO, doladěný na **RDD2022** a pak na vlastních záběrech.
Metriky: mAP50, mAP50-95, precision, recall.

**Runtime → Change runtime type → T4 GPU.**

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE_DIR = '/content/drive/MyDrive/motorcam'
REPO_URL = 'https://github.com/MiniGue1/motorcam.git'   # u soukromého repa: https://<token>@github.com/...

import os
%cd /content
if not os.path.exists('motorcam'):
    !git clone -q $REPO_URL motorcam
%cd /content/motorcam
!pip install -q -r requirements.txt
# archivy, vlastní data, modely a výsledky trvale na Google Drive
for d in ['data/raw/rdd2022/archives', 'data/raw/rscd/archives', 'data/custom', 'models', 'results']:
    os.makedirs(f'{DRIVE_DIR}/{d}', exist_ok=True); os.makedirs(os.path.dirname(d) or '.', exist_ok=True)
    if not os.path.islink(d):
        !rm -rf $d
        os.symlink(f'{DRIVE_DIR}/{d}', d)
import torch; print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NENÍ – Runtime → Change runtime type → T4 GPU')

In [ ]:
# data z fáze 1 (archivy už jsou na Drive – stáhne jen chybějící, pak rozbalí a převede)
!python -m src.dataset.download rdd2022
!python -m src.dataset.rdd_to_yolo

In [ ]:
import os
if os.path.exists('data/custom/labelstudio_export.json'):
    !python -m src.dataset.import_annotations labelstudio data/custom/labelstudio_export.json

## 1) Jen veřejná data (RDD2022)
60 epoch, 640 px ≈ 1,5–2,5 h na T4 (Colab free má limit ~3–4 h/den – případně `--epochs 40`).

In [ ]:
!python -m src.models.detector train --sources data/processed/rdd2022_yolo --name det_public \
    --epochs 60 --batch 32 --imgsz 640

In [ ]:
from IPython.display import Image, display
import pandas as pd
display(pd.read_csv('results/tables/det_public_val_metriky.csv'))
for f in ['results/figures/det_public_val_PR.png', 'models/det_public/results.png']:
    if os.path.exists(f): display(Image(f))

## 2) Doladění na vlastních záběrech
RDD2022 + vlastní snímky (3× častěji), start z `det_public`.

In [ ]:
if os.path.exists('data/processed/custom_yolo/images/train'):
    !python -m src.models.detector train --sources data/processed/rdd2022_yolo data/processed/custom_yolo \
        --custom-repeat 3 --init models/det_public/weights/best.pt --name det_finetuned --epochs 30 --batch 32
    display(pd.read_csv('results/tables/det_finetuned_test_metriky.csv'))